### Initialization

In [0]:

import pyspark.sql.functions as F
from pyspark.sql.types import StringType
from pyspark.sql.functions import trim, col

### Read Bronze table

In [0]:
df = spark.table("workspace.bronze.erp_px_cat_g1v2")
print(f"print all rows {df.count()}")
#display(df)

### Silver Transformations

### Trimming

In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType,StringType):
        df = df.withColumn(field.name,trim(col(field.name)))

### Normalize Maintenance Flag to Boolean

In [0]:
df = df.withColumn(
    "MAINTENANCE",
    F.when(F.upper(F.col("MAINTENANCE")) == "YES",F.lit(True))
    .when(F.upper(F.col("MAINTENANCE"))== "NO",F.lit(False))
    .otherwise(None)
)

### Renaming Columns

In [0]:
RENAME_MAP = {
    "ID": "category_id",
    "CAT": "category",
    "SUBCAT": "subcategory",
    "MAINTENANCE": "maintenance_flag"
}
for old_name, new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)
     

### Sanity checks of dataframe

In [0]:
print("Sample Data :")
df.limit(10).display()
print("\n Category Distibution:")
df.groupBy("category","subcategory").count().orderBy("category").display()
print("\nMaintence flag distibution")


### Writing Silver Table

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable("workspace.silver.erp_px_cat_g1v2")
print(f"Written {df.count()} rows to workspace.silver.erp_px_cat_g1v2")

### Sanity checks of silver table

In [0]:

%sql
SELECT * FROM workspace.silver.erp_px_cat_g1v2 LIMIT 10